# 第一段階：角度較正前後の誤差チャネル比較

有限 Lamb–Dicke MS ゲートについて、矩形パルスの振幅を \(h_{XX}=0\) に近づけるよう較正し、その前後で**誤差チャネル全体**を比較する。

中心となる問いは次の三つである。

1. 角度較正により Hamiltonian 型の \(h_{XX}\) は実際に除去されるか。
2. 較正後も \(\gamma_{XX}\)、\(\gamma_{\rm col}\)、平均ゲート infidelity は残るか。
3. 三成分生成子
   
   \[
   K_3=h_{XX}\mathcal H[XX]+\gamma_{XX}\mathcal D[XX]
       +\gamma_{\rm col}\mathcal D[IX+XI]
   \]
   
   は較正前後の full QPT 生成子をどこまで説明できるか。

このノートの較正は各 \(\bar n\) で独立に行うため、角度誤差を最もよく除いた best-case 比較である。一つの温度で得た較正値を別の温度へ移す検証は次段階とする。

## 0. 判定の考え方

- \(h_{XX}\) が大きく減る一方で infidelity があまり減らなければ、単一角度較正では除去できない誤差床の候補となる。
- \(\gamma_{XX}\) と \(\gamma_{\rm col}\) は較正後に必ず不変とは限らない。振幅変更後の QPT から毎回取り直す。
- \(\mathcal D[IX+XI]\) なら完全な H/S/C/A 分解で \(S_{IX}=S_{XI}=C_{IX,XI}\) が期待される。その係数の spread を collective-X 仮定の診断に使う。
- 三成分残差が大きければ、三成分は説明用の低次元近似にすぎず、チャネルの普遍的・厳密な記述とはみなさない。

以下の閾値は普遍的な物理定数ではなく、この解析で仮説を棄却しやすくするための編集可能な operational criterion である。

In [2]:
from pathlib import Path
import importlib
import json
import os
import sys

import matplotlib.pyplot as plt
from matplotlib.colors import LogNorm
import numpy as np
import pandas as pd
import scipy.linalg
import scipy.optimize
from IPython.display import Markdown, display


def find_project_root(start=None):
    current = Path(start or Path.cwd()).resolve()
    for candidate in (current, *current.parents):
        if (candidate / "chi_error_nbar_workflow.py").exists():
            return candidate
    raise FileNotFoundError("chi_error_nbar_workflow.py がある project root を見つけられません")


PROJECT_ROOT = find_project_root()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))
os.chdir(PROJECT_ROOT)

import chi_error_nbar_workflow as workflow
import chi_error_nbar_stages as stages
import drive_calibration_qpt_analysis as qpt_analysis
import ms_gate_functions as mg
import noise_error_structure as noise_structure

workflow = importlib.reload(workflow)
stages = importlib.reload(stages)
qpt_analysis = importlib.reload(qpt_analysis)
noise_structure = importlib.reload(noise_structure)

pd.set_option("display.max_columns", 100)
pd.set_option("display.float_format", lambda value: f"{value:.6g}")
print(f"PROJECT_ROOT = {PROJECT_ROOT}")

PROJECT_ROOT = /workspace


## 1. 設定

既定では QPT を再計算せず、既存キャッシュだけを読む。不足キャッシュを計算するときだけ RUN_QPT を True にする。FORCE_RECOMPUTE は既存キャッシュまで上書きするため、通常は False のままにする。

In [3]:
CONFIG = workflow.default_config()
CONFIG["OUTPUT_DIR"] = str(PROJECT_ROOT / "results" / "chi_error_element_fit")


WORKER_COUNT = 24
if WORKER_COUNT < 1:
    raise ValueError("WORKER_COUNT must be at least 1")
CONFIG["PARALLEL_WORKERS"] = WORKER_COUNT
CONFIG["FAST_PROCESS_WORKERS"] = WORKER_COUNT
CONFIG["SIMULATION_PARAMS"]["parallel_workers"] = WORKER_COUNT

NBAR_VALUES = [0.01, 1.0, 2.0, 4.0, 6.0, 8.0, 10.0]
RUN_QPT = True
FORCE_RECOMPUTE = False
MAX_FEEDBACK_ITERATIONS = 2
SELECTED_NBAR_FOR_CHI = 4.0

HXX_TOL_RAD_PER_GATE = 2e-3
MODEL_RESIDUAL_FRACTION_MAX = 0.25
COLLECTIVE_SPREAD_RELATIVE_MAX = 0.25
STRONG_HXX_REDUCTION_FACTOR = 10.0
INFIDELITY_FLOOR_REMAINING_MIN = 0.50

ANALYSIS_DIR = PROJECT_ROOT / "results" / "calibration_channel_comparison"
ANALYSIS_DIR.mkdir(parents=True, exist_ok=True)

workflow.validate_config(CONFIG)
display(pd.Series({
    "worker_count": WORKER_COUNT,
    "nbar_values": NBAR_VALUES,
    "run_qpt": RUN_QPT,
    "force_recompute": FORCE_RECOMPUTE,
    "max_feedback_iterations": MAX_FEEDBACK_ITERATIONS,
    "analysis_dir": str(ANALYSIS_DIR),
}, name="value").to_frame())

,value
worker_count,24
nbar_values,"[0.01, 1.0, 2.0, 4.0, 6.0, 8.0, 10.0]"
run_qpt,True
force_recompute,False
max_feedback_iterations,2
analysis_dir,/workspace/results/calibration_channel_comparison


## 2. 振幅較正と full-Hamiltonian QPT

較正前の \(h_{XX}\) から

\[
A_{k+1}=A_k\sqrt{\frac{\pi/4}{\pi/4-h_{XX}^{(k)}}}
\]

を予測値として用い、その振幅で full-Hamiltonian QPT をやり直す。高次補正により応答は厳密な二次則ではないため、残留 \(h_{XX}\) が閾値を超えた場合だけフィードバックを繰り返す。

In [ ]:
drive_result = stages.run_drive_feedback_stage(
    CONFIG,
    NBAR_VALUES,
    run_qpt=RUN_QPT,
    force_recompute=FORCE_RECOMPUTE,
    max_feedback_iterations=MAX_FEEDBACK_ITERATIONS,
)

print(json.dumps(drive_result["status"], indent=2, ensure_ascii=False, default=str))
if drive_result["status"].get("pending"):
    display(pd.DataFrame(drive_result["status"]["pending"]))

drive_summary = drive_result["summary"].copy()
if drive_summary.empty:
    raise RuntimeError(
        "較正後 QPT がありません。不足点を計算する場合は RUN_QPT=True にしてこのセルを再実行してください。"
    )

display(drive_summary[[
    "n_bar", "iteration", "A_factor",
    "h_XX_before_rad_per_gate", "h_XX_after_rad_per_gate",
    "average_infidelity_before", "average_infidelity_after",
    "h_XX_converged",
]])

## 3. チャネルから三成分生成子を抽出する

各 QPT の \(\chi\) を CPTP 射影し、Pauli transfer matrix \(R\) の主値対数 \(K=\log R\) を取る。Hamiltonian 部分には \(\mathcal H[XX]\)、対称部分には \(\mathcal D[XX]\) と \(\mathcal D[IX+XI]\) を最小二乗で当てる。二つの散逸率には非負制約を課す。

同時に完全な H/S/C/A 基底でも分解し、collective-X の三係数 \(S_{IX}, S_{XI}, C_{IX,XI}\) が等しいかを独立に確認する。

In [ ]:
def legacy_nbar_stem(n_bar):
    return str(float(n_bar)).replace("-", "m").replace(".", "p")


def baseline_chi_path(n_bar):
    return (
        Path(CONFIG["OUTPUT_DIR"])
        / "advanced_publication_validation"
        / "cptp_projection"
        / f"cptp_chi_nbar_{legacy_nbar_stem(n_bar)}.npz"
    )


def resolve_project_path(path_like):
    path = Path(path_like)
    return path if path.is_absolute() else PROJECT_ROOT / path


def load_trace_normalized_chi(path_like):
    path = resolve_project_path(path_like)
    if not path.exists():
        raise FileNotFoundError(path)
    with np.load(path, allow_pickle=False) as data:
        chi = np.asarray(data["chi_trace_normalized"], dtype=complex)
    return chi, path


def build_three_component_bases():
    labels, hamiltonian_bases, _, dissipator_design = qpt_analysis._generator_design_data()
    pauli_labels = list(labels[1:])
    pauli_qobjs = tuple(mg.two_qubit_pauli_basis())
    pauli_by_label = dict(zip(labels, pauli_qobjs))
    h_xx = np.asarray(hamiltonian_bases["XX"], dtype=float)
    d_xx = dissipator_design[:, pauli_labels.index("XX")].reshape(h_xx.shape)
    jump = pauli_by_label["IX"] + pauli_by_label["XI"]
    jump_squared = jump * jump
    d_col = qpt_analysis._action_to_ptm(
        lambda rho: jump * rho * jump - 0.5 * (jump_squared * rho + rho * jump_squared),
        pauli_qobjs,
    )
    return {"H_XX": h_xx, "D_XX": d_xx, "D_col": d_col}


THREE_COMPONENT_BASES = build_three_component_bases()


def taxonomy_lookup(taxonomy):
    return {
        f"{sector}:{mode}": float(value)
        for (sector, mode), value in zip(taxonomy["metadata"], taxonomy["coefficients"])
    }


def analyze_channel(chi):
    superoperator, projected_chi, projection_status = (
        qpt_analysis.project_trace_normalized_chi_to_cptp(
            chi,
            tolerance=CONFIG.get("CPTP_TOLERANCE", 1e-11),
            max_iterations=CONFIG.get("CPTP_MAX_ITERATIONS", 5000),
        )
    )
    ptm = np.asarray(mg.superoperator_to_ptm(superoperator), dtype=complex)
    generator_complex = scipy.linalg.logm(ptm)
    generator = np.real(generator_complex)

    h_basis = THREE_COMPONENT_BASES["H_XX"]
    d_xx_basis = THREE_COMPONENT_BASES["D_XX"]
    d_col_basis = THREE_COMPONENT_BASES["D_col"]

    skew = 0.5 * (generator - generator.T)
    h_xx = float(np.linalg.lstsq(h_basis.reshape(-1, 1), skew.reshape(-1), rcond=None)[0][0])
    h_fit = h_xx * h_basis
    symmetric_remaining = 0.5 * ((generator - h_fit) + (generator - h_fit).T)
    dissipator_design = np.column_stack([d_xx_basis.reshape(-1), d_col_basis.reshape(-1)])
    gamma, gamma_nnls_residual = scipy.optimize.nnls(
        dissipator_design, symmetric_remaining.reshape(-1)
    )
    gamma_xx, gamma_col = map(float, gamma)
    model_generator = h_fit + gamma_xx * d_xx_basis + gamma_col * d_col_basis

    taxonomy = noise_structure._decompose_generator_taxonomy(generator)
    coefficients = taxonomy_lookup(taxonomy)
    collective_coefficients = np.array([
        coefficients["S:IX"],
        coefficients["S:XI"],
        coefficients["C:IX,XI"],
    ])
    collective_mean = float(np.mean(collective_coefficients))
    collective_spread = float(np.ptp(collective_coefficients))

    labels = list(qpt_analysis._generator_design_data()[0])
    ii_index = labels.index("II")
    xx_index = labels.index("XX")
    generator_norm = float(np.linalg.norm(generator))
    model_residual_norm = float(np.linalg.norm(generator - model_generator))
    ptm_real = np.real(ptm)

    scalars = {
        "h_XX_rad_per_gate": h_xx,
        "gamma_XX_per_gate": gamma_xx,
        "gamma_col_per_gate": gamma_col,
        "average_infidelity": 4.0 / 5.0 * (1.0 - float(np.real(projected_chi[ii_index, ii_index]))),
        "abs_chi_II_XX": float(abs(projected_chi[ii_index, xx_index])),
        "chi_XX_XX": float(np.real(projected_chi[xx_index, xx_index])),
        "generator_frobenius_norm": generator_norm,
        "three_component_residual_norm": model_residual_norm,
        "three_component_residual_fraction": model_residual_norm / max(generator_norm, 1e-15),
        "three_component_channel_error_fraction": float(
            np.linalg.norm(scipy.linalg.expm(model_generator) - ptm_real)
            / max(np.linalg.norm(ptm_real), 1e-15)
        ),
        "gamma_nnls_residual": float(gamma_nnls_residual),
        "taxonomy_H_XX": coefficients["H:XX"],
        "taxonomy_S_XX": coefficients["S:XX"],
        "taxonomy_S_IX": coefficients["S:IX"],
        "taxonomy_S_XI": coefficients["S:XI"],
        "taxonomy_C_IX_XI": coefficients["C:IX,XI"],
        "collective_x_coefficient_mean": collective_mean,
        "collective_x_coefficient_spread": collective_spread,
        "collective_x_relative_spread": collective_spread / max(abs(collective_mean), 1e-15),
        "taxonomy_reconstruction_residual_fraction": float(
            np.linalg.norm(taxonomy["residual"]) / max(generator_norm, 1e-15)
        ),
        "generator_imaginary_fraction": float(
            np.linalg.norm(np.imag(generator_complex)) / max(generator_norm, 1e-15)
        ),
        "cptp_min_choi_eigenvalue": float(projection_status["min_choi_eigenvalue"]),
        "cptp_tp_frobenius_error": float(projection_status["tp_frobenius_error"]),
    }
    return {
        "scalars": scalars,
        "chi": projected_chi,
        "ptm": ptm_real,
        "generator": generator,
        "model_generator": model_generator,
    }

## 4. 較正前後を同じ規約で再解析する

既存 summary の数値をそのまま横に並べるだけでなく、較正前後の \(\chi\) を同じ CPTP 射影・matrix-log・三成分 fit に通す。これにより比較時の解析規約を揃える。

In [ ]:
records = []
channel_objects = {}

for n_bar in NBAR_VALUES:
    matched = drive_summary.loc[np.isclose(drive_summary["n_bar"].astype(float), n_bar)]
    if matched.empty:
        raise ValueError(f"n_bar={n_bar:g} の較正結果がありません")
    calibrated_row = matched.sort_values("iteration").iloc[-1]
    conditions = [
        ("before", baseline_chi_path(n_bar), 1.0, 0),
        (
            "after",
            calibrated_row["cache_path"],
            float(calibrated_row["A_factor"]),
            int(calibrated_row["iteration"]),
        ),
    ]

    for condition, source_path, amplitude_factor, iteration in conditions:
        chi, resolved_path = load_trace_normalized_chi(source_path)
        analyzed = analyze_channel(chi)
        channel_objects[(float(n_bar), condition)] = analyzed
        records.append({
            "n_bar": float(n_bar),
            "condition": condition,
            "A_factor": amplitude_factor,
            "feedback_iteration": iteration,
            "source_path": str(resolved_path),
            **analyzed["scalars"],
        })

comparison_long = pd.DataFrame(records).sort_values(["n_bar", "condition"]).reset_index(drop=True)
comparison_long_path = ANALYSIS_DIR / "channel_comparison_long.csv"
comparison_long.to_csv(comparison_long_path, index=False)

before = comparison_long.query("condition == 'before'").drop(columns="condition")
after = comparison_long.query("condition == 'after'").drop(columns="condition")
paired = before.merge(after, on="n_bar", suffixes=("_before", "_after"))

paired["abs_h_XX_reduction_factor"] = (
    np.abs(paired["h_XX_rad_per_gate_before"])
    / np.maximum(np.abs(paired["h_XX_rad_per_gate_after"]), 1e-15)
)
paired["infidelity_remaining_fraction"] = (
    paired["average_infidelity_after"]
    / np.maximum(paired["average_infidelity_before"], 1e-15)
)
paired["infidelity_reduction_percent"] = 100.0 * (1.0 - paired["infidelity_remaining_fraction"])
paired["gamma_XX_change"] = paired["gamma_XX_per_gate_after"] - paired["gamma_XX_per_gate_before"]
paired["gamma_col_change"] = paired["gamma_col_per_gate_after"] - paired["gamma_col_per_gate_before"]
paired["h_XX_converged"] = np.abs(paired["h_XX_rad_per_gate_after"]) <= HXX_TOL_RAD_PER_GATE

paired_path = ANALYSIS_DIR / "channel_comparison_paired.csv"
paired.to_csv(paired_path, index=False)

ordered_nbars = np.asarray(NBAR_VALUES, dtype=float)
chi_before_stack = np.stack([channel_objects[(float(n), "before")]["chi"] for n in ordered_nbars])
chi_after_stack = np.stack([channel_objects[(float(n), "after")]["chi"] for n in ordered_nbars])
np.savez_compressed(
    ANALYSIS_DIR / "projected_chi_before_after.npz",
    n_bar=ordered_nbars,
    chi_before=chi_before_stack,
    chi_after=chi_after_stack,
)

display(paired[[
    "n_bar", "A_factor_after",
    "h_XX_rad_per_gate_before", "h_XX_rad_per_gate_after", "abs_h_XX_reduction_factor",
    "gamma_XX_per_gate_before", "gamma_XX_per_gate_after",
    "gamma_col_per_gate_before", "gamma_col_per_gate_after",
    "average_infidelity_before", "average_infidelity_after", "infidelity_reduction_percent",
    "three_component_residual_fraction_before", "three_component_residual_fraction_after",
]])
print(f"Saved: {comparison_long_path}")
print(f"Saved: {paired_path}")

## 5. 主結果：較正で消える成分と残る成分

左上で角度較正の成立を、中央二枚で stochastic 成分の残留を、右上でチャネル全体の infidelity を見る。下段右は、三成分を使うこと自体の妥当性診断である。

In [ ]:
figure, axes = plt.subplots(2, 3, figsize=(16.0, 9.0))

panels = [
    (axes[0, 0], "h_XX_rad_per_gate", r"$h_{XX}$ (rad/gate)", False),
    (axes[0, 1], "gamma_XX_per_gate", r"$\gamma_{XX}$ (1/gate)", True),
    (axes[0, 2], "gamma_col_per_gate", r"$\gamma_{\rm col}$ (1/gate)", True),
    (axes[1, 0], "average_infidelity", "Average infidelity", True),
    (axes[1, 1], "three_component_residual_fraction", r"$\|K-K_3\|_F/\|K\|_F$", False),
]

for axis, column, ylabel, log_scale in panels:
    for condition, label, marker in [("before", "before calibration", "o"), ("after", "after calibration", "s")]:
        subset = comparison_long.query("condition == @condition").sort_values("n_bar")
        values = subset[column].to_numpy(float)
        if log_scale:
            values = np.maximum(np.abs(values), 1e-16)
        axis.plot(subset["n_bar"], values, marker=marker, linewidth=2.0, label=label)
    if log_scale:
        axis.set_yscale("log")
    axis.set_xlabel(r"Mean phonon number $\bar n$")
    axis.set_ylabel(ylabel)
    axis.grid(True, which="both", alpha=0.28)

axes[0, 0].axhline(0.0, color="black", linewidth=0.8)
axes[0, 0].axhspan(-HXX_TOL_RAD_PER_GATE, HXX_TOL_RAD_PER_GATE, color="tab:green", alpha=0.10)
axes[0, 0].legend()

axes[1, 2].plot(paired["n_bar"], paired["abs_h_XX_reduction_factor"], "o-", label=r"$|h_{XX}|$ reduction")
axes[1, 2].plot(paired["n_bar"], 1.0 / np.maximum(paired["infidelity_remaining_fraction"], 1e-15), "s-", label="infidelity reduction")
axes[1, 2].set_yscale("log")
axes[1, 2].set_xlabel(r"Mean phonon number $\bar n$")
axes[1, 2].set_ylabel("Reduction factor")
axes[1, 2].grid(True, which="both", alpha=0.28)
axes[1, 2].legend()

figure.suptitle("Error channel before and after XX-angle calibration", y=1.01, fontsize=15)
figure.tight_layout()
main_figure_png = ANALYSIS_DIR / "channel_before_after_main.png"
main_figure_pdf = ANALYSIS_DIR / "channel_before_after_main.pdf"
figure.savefig(main_figure_png, dpi=300, bbox_inches="tight")
figure.savefig(main_figure_pdf, bbox_inches="tight")
display(figure)
plt.close(figure)
print(f"Saved: {main_figure_png}")

## 6. collective-X 仮定と三成分近似の診断

restricted fit の \(\gamma_{\rm col}\) が得られても、それだけでは \(\mathcal D[IX+XI]\) が正しいとは言えない。完全分解の \(S_{IX},S_{XI},C_{IX,XI}\) の一致と、full generator に対する残差を同時に確認する。

In [ ]:
collective_columns = [
    "n_bar", "condition", "gamma_col_per_gate",
    "taxonomy_S_IX", "taxonomy_S_XI", "taxonomy_C_IX_XI",
    "collective_x_relative_spread", "three_component_residual_fraction",
    "three_component_channel_error_fraction",
]
collective_diagnostic = comparison_long[collective_columns].copy()
collective_diagnostic_path = ANALYSIS_DIR / "three_component_model_diagnostics.csv"
collective_diagnostic.to_csv(collective_diagnostic_path, index=False)
display(collective_diagnostic)

figure, axes = plt.subplots(1, 2, figsize=(13.0, 4.8))
after_rows = comparison_long.query("condition == 'after'").sort_values("n_bar")
for column, label, marker in [
    ("taxonomy_S_IX", r"$S_{IX}$", "o"),
    ("taxonomy_S_XI", r"$S_{XI}$", "s"),
    ("taxonomy_C_IX_XI", r"$C_{IX,XI}$", "^"),
]:
    axes[0].plot(after_rows["n_bar"], after_rows[column], marker=marker, label=label)
axes[0].set_xlabel(r"Mean phonon number $\bar n$")
axes[0].set_ylabel("Taxonomy coefficient (1/gate)")
axes[0].set_title("Collective-X closure after calibration")
axes[0].grid(True, alpha=0.28)
axes[0].legend()

for condition, label, marker in [("before", "before", "o"), ("after", "after", "s")]:
    subset = comparison_long.query("condition == @condition").sort_values("n_bar")
    axes[1].plot(
        subset["n_bar"], subset["three_component_residual_fraction"],
        marker=marker, label=label,
    )
axes[1].axhline(MODEL_RESIDUAL_FRACTION_MAX, color="tab:red", linestyle="--", label="operational threshold")
axes[1].set_xlabel(r"Mean phonon number $\bar n$")
axes[1].set_ylabel(r"$\|K-K_3\|_F/\|K\|_F$")
axes[1].set_title("Three-component model adequacy")
axes[1].grid(True, alpha=0.28)
axes[1].legend()

figure.tight_layout()
diagnostic_png = ANALYSIS_DIR / "three_component_diagnostics.png"
diagnostic_pdf = ANALYSIS_DIR / "three_component_diagnostics.pdf"
figure.savefig(diagnostic_png, dpi=300, bbox_inches="tight")
figure.savefig(diagnostic_pdf, bbox_inches="tight")
display(figure)
plt.close(figure)
print(f"Saved: {collective_diagnostic_path}")

## 7. 代表温度における \(\chi\) 行列そのものの比較

三成分だけを見て結論を固定しないため、代表的な \(\bar n\) について CPTP 射影後の \(|\chi|\) を直接可視化する。右端は較正による変化量である。

In [ ]:
available_nbars = np.asarray(sorted(comparison_long["n_bar"].unique()), dtype=float)
selected_nbar = float(available_nbars[np.argmin(np.abs(available_nbars - SELECTED_NBAR_FOR_CHI))])
chi_before = channel_objects[(selected_nbar, "before")]["chi"]
chi_after = channel_objects[(selected_nbar, "after")]["chi"]
chi_delta = chi_after - chi_before

pauli_labels = list(qpt_analysis._generator_design_data()[0])
matrices = [np.abs(chi_before), np.abs(chi_after), np.abs(chi_delta)]
titles = ["Before calibration", "After calibration", r"$|\Delta\chi|$"]
vmax = max(float(np.max(matrix)) for matrix in matrices)
vmin = max(vmax * 1e-7, 1e-12)

figure, axes = plt.subplots(1, 3, figsize=(17.0, 5.2))
for axis, matrix, title in zip(axes, matrices, titles):
    image = axis.imshow(matrix, origin="lower", cmap="magma", norm=LogNorm(vmin=vmin, vmax=vmax))
    axis.set_title(title)
    axis.set_xticks(range(len(pauli_labels)), pauli_labels, rotation=90, fontsize=8)
    axis.set_yticks(range(len(pauli_labels)), pauli_labels, fontsize=8)
    axis.set_xlabel("input Pauli index")
axes[0].set_ylabel("output Pauli index")
figure.colorbar(image, ax=axes, fraction=0.025, pad=0.02, label=r"$|\chi_{P,Q}|$")
figure.suptitle(rf"Trace-normalized error $\chi$ at $\bar n={selected_nbar:g}$", y=1.02)
chi_figure_png = ANALYSIS_DIR / f"chi_before_after_nbar_{legacy_nbar_stem(selected_nbar)}.png"
chi_figure_pdf = ANALYSIS_DIR / f"chi_before_after_nbar_{legacy_nbar_stem(selected_nbar)}.pdf"
figure.savefig(chi_figure_png, dpi=300, bbox_inches="tight")
figure.savefig(chi_figure_pdf, bbox_inches="tight")
display(figure)
plt.close(figure)
print(f"Saved: {chi_figure_png}")

## 8. 仮説の機械的チェックと結論候補

ここでは結論を自動的に証明するのではなく、どの主張が数値に支持され、どこが反証されたかを整理する。特に三成分 model の不適合を隠さない。

In [ ]:
after_rows = comparison_long.query("condition == 'after'")
median_h_reduction = float(np.median(paired["abs_h_XX_reduction_factor"]))
median_infidelity_remaining = float(np.median(paired["infidelity_remaining_fraction"]))
max_after_residual = float(after_rows["three_component_residual_fraction"].max())
max_after_collective_spread = float(after_rows["collective_x_relative_spread"].max())

decision_rows = [
    {
        "question": "角度較正は収束したか",
        "criterion": f"全点で |h_XX| <= {HXX_TOL_RAD_PER_GATE:.2g}",
        "value": bool(paired["h_XX_converged"].all()),
        "supported": bool(paired["h_XX_converged"].all()),
    },
    {
        "question": "h_XX は強く除去されたか",
        "criterion": f"median reduction >= {STRONG_HXX_REDUCTION_FACTOR:g}",
        "value": median_h_reduction,
        "supported": median_h_reduction >= STRONG_HXX_REDUCTION_FACTOR,
    },
    {
        "question": "角度較正後の infidelity floor は残るか",
        "criterion": f"median remaining fraction >= {INFIDELITY_FLOOR_REMAINING_MIN:g}",
        "value": median_infidelity_remaining,
        "supported": median_infidelity_remaining >= INFIDELITY_FLOOR_REMAINING_MIN,
    },
    {
        "question": "三成分 model は全点で十分か",
        "criterion": f"max residual fraction <= {MODEL_RESIDUAL_FRACTION_MAX:g}",
        "value": max_after_residual,
        "supported": max_after_residual <= MODEL_RESIDUAL_FRACTION_MAX,
    },
    {
        "question": "collective-X の係数閉包は全点で良いか",
        "criterion": f"max relative spread <= {COLLECTIVE_SPREAD_RELATIVE_MAX:g}",
        "value": max_after_collective_spread,
        "supported": max_after_collective_spread <= COLLECTIVE_SPREAD_RELATIVE_MAX,
    },
]
decision_table = pd.DataFrame(decision_rows)
decision_path = ANALYSIS_DIR / "hypothesis_checks.csv"
decision_table.to_csv(decision_path, index=False)
display(decision_table)

floor_message = (
    "角度誤差を強く抑えてもチャネル infidelity の相当部分が残る。単一角度較正で除去できない誤差床、という主張の候補になる。"
    if median_h_reduction >= STRONG_HXX_REDUCTION_FACTOR
    and median_infidelity_remaining >= INFIDELITY_FLOOR_REMAINING_MIN
    else "このデータだけでは、単一角度較正後の誤差床という主張はまだ十分に支持されない。"
)
model_message = (
    "三成分 model は設定した残差基準を全点で満たす。"
    if max_after_residual <= MODEL_RESIDUAL_FRACTION_MAX
    else "三成分 model は少なくとも一部の温度で残差基準を破るため、残差の主要 H/S/C/A 成分を追加調査する必要がある。"
)

display(Markdown(
    "### 暫定結論\n\n"
    f"- {floor_message}\n"
    f"- {model_message}\n"
    "- ここで示すのは数値 QPT 内での calibration-aware comparison であり、全 MS ゲートへの普遍性や実験での因果を証明するものではない。\n"
    "- 論文の中心図にする前に、数値収束、QPT 不確かさ、固定較正値の温度間 transfer、実験 process matrix での再現を追加する。"
))

## 9. 出力一覧

CSV は論文表・再解析用、NPZ は較正前後の \(\chi\) 行列、PNG/PDF は図用である。

In [ ]:
manifest = {
    "analysis": "XX-angle calibration before/after channel comparison",
    "error_channel_convention": CONFIG.get("ERROR_CHANNEL_CONVENTION"),
    "calibration_protocol": "independent hXX feedback at each n_bar",
    "n_bar_values": [float(value) for value in NBAR_VALUES],
    "worker_count": int(WORKER_COUNT),
    "run_qpt": bool(RUN_QPT),
    "force_recompute": bool(FORCE_RECOMPUTE),
    "outputs": sorted(str(path.relative_to(PROJECT_ROOT)) for path in ANALYSIS_DIR.iterdir()),
    "operational_thresholds": {
        "hxx_tolerance_rad_per_gate": HXX_TOL_RAD_PER_GATE,
        "model_residual_fraction_max": MODEL_RESIDUAL_FRACTION_MAX,
        "collective_spread_relative_max": COLLECTIVE_SPREAD_RELATIVE_MAX,
        "strong_hxx_reduction_factor": STRONG_HXX_REDUCTION_FACTOR,
        "infidelity_floor_remaining_min": INFIDELITY_FLOOR_REMAINING_MIN,
    },
}
manifest_path = ANALYSIS_DIR / "manifest.json"
manifest_path.write_text(json.dumps(manifest, indent=2, ensure_ascii=False), encoding="utf-8")
display(pd.DataFrame({"output": manifest["outputs"]}))
print(f"Saved: {manifest_path}")